# CSL7110 Assignment 2  
## MinHashing and Locality Sensitive Hashing

**Name:** Kunal Mishra  
**Roll Number:** M25CSA036  
**Course:** Machine Learning with Big Data


In [2]:
import os
os.listdir()

['minhash.zip',
 '.DS_Store',
 'minhash',
 'M25CSA036_CSL7110_Assignment.ipynb.ipynb',
 '.ipynb_checkpoints',
 'CSL7110_Assignment_2.pdf']

In [3]:
base_path = "minhash"

def read_file(filename):
    with open(os.path.join(base_path, filename), 'r') as f:
        return f.read().strip()

D1 = read_file("D1.txt")
D2 = read_file("D2.txt")
D3 = read_file("D3.txt")
D4 = read_file("D4.txt")

documents = {
    "D1": D1,
    "D2": D2,
    "D3": D3,
    "D4": D4
}

print("Documents loaded successfully!")

Documents loaded successfully!


In [4]:
for name, text in documents.items():
    print(name, ":", text)

D1 : apple ceo tim cook is spending some time in canada this week and yesterday he attended a hockey game and visited the eaton centre apple store in toronto cook today stopped by the offices of canadian ecommerce platform shopify where he spoke to the financial post about augmented reality apps and the homepod on the topic of the homepod cook said that apples deep integration between hardware and software will help to differentiate the smart speaker from competing products like amazons alexa and the google home competition makes all of us better and i welcome it cook said but if you are both trying to license something and compete with your licensees this is a difficult model and it remains to be seen if it can be successful or not cook also said a quality very immersive audio experience was one thing missing from the smart speaker market which apple is aiming to fix music deserves that kind of quality as opposed to some kind of squeaky sound he said the homepod which at in the united

## Question 1: K-Grams and Jaccard Similarity
In this section we generate character 2-grams, character 3-grams and word 2-grams for four documents and compute Jaccard similarity between all document pairs.

In [5]:
def char_kgrams(text, k):
    return set(text[i:i+k] for i in range(len(text) - k + 1))

In [6]:
char3 = {}

for name, text in documents.items():
    char3[name] = char_kgrams(text, 3)

print("Character 3-grams created successfully.")

Character 3-grams created successfully.


In [7]:
for name in char3:
    print(f"{name} - Number of 3-grams:", len(char3[name]))

D1 - Number of 3-grams: 765
D2 - Number of 3-grams: 762
D3 - Number of 3-grams: 828
D4 - Number of 3-grams: 698


In [8]:
char2 = {}

for name, text in documents.items():
    char2[name] = char_kgrams(text, 2)

print("Character 2-grams created successfully.")

Character 2-grams created successfully.


In [9]:
for name in char2:
    print(f"{name} - Number of 2-grams:", len(char2[name]))

D1 - Number of 2-grams: 263
D2 - Number of 2-grams: 262
D3 - Number of 2-grams: 269
D4 - Number of 2-grams: 255


In [10]:
def word_kgrams(text, k):
    words = text.split()
    return set(" ".join(words[i:i+k]) for i in range(len(words) - k + 1))

In [11]:
word2 = {}

for name, text in documents.items():
    word2[name] = word_kgrams(text, 2)

print("Word 2-grams created successfully.")

Word 2-grams created successfully.


In [12]:
for name in word2:
    print(f"{name} - Number of word 2-grams:", len(word2[name]))

D1 - Number of word 2-grams: 279
D2 - Number of word 2-grams: 278
D3 - Number of word 2-grams: 337
D4 - Number of word 2-grams: 232


In [13]:
def jaccard(A, B):
    return len(A & B) / len(A | B)

In [14]:
import itertools

pairs = list(itertools.combinations(documents.keys(), 2))

print("Character 2-gram Similarities")
for d1, d2 in pairs:
    print(f"{d1}-{d2}: {jaccard(char2[d1], char2[d2])}")

print("\nCharacter 3-gram Similarities")
for d1, d2 in pairs:
    print(f"{d1}-{d2}: {jaccard(char3[d1], char3[d2])}")

print("\nWord 2-gram Similarities")
for d1, d2 in pairs:
    print(f"{d1}-{d2}: {jaccard(word2[d1], word2[d2])}")

Character 2-gram Similarities
D1-D2: 0.9811320754716981
D1-D3: 0.8156996587030717
D1-D4: 0.6444444444444445
D2-D3: 0.8
D2-D4: 0.6412698412698413
D3-D4: 0.6529968454258676

Character 3-gram Similarities
D1-D2: 0.977979274611399
D1-D3: 0.5803571428571429
D1-D4: 0.3050847457627119
D2-D3: 0.5680473372781065
D2-D4: 0.30590339892665475
D3-D4: 0.31212381771281167

Word 2-gram Similarities
D1-D2: 0.9407665505226481
D1-D3: 0.18234165067178504
D1-D4: 0.03024193548387097
D2-D3: 0.1736641221374046
D2-D4: 0.030303030303030304
D3-D4: 0.01607142857142857


## Question 2: MinHashing
MinHash signatures are generated using different numbers of hash functions. The approximate similarity is compared with the exact Jaccard similarity.

In [15]:
m = 20011

In [16]:
import hashlib

def hash_kgram(kgram):
    return int(hashlib.md5(kgram.encode()).hexdigest(), 16)

In [17]:
import random

def generate_hash_functions(t):
    hash_funcs = []
    for _ in range(t):
        a = random.randint(1, m-1)
        b = random.randint(0, m-1)
        hash_funcs.append((a, b))
    return hash_funcs

In [18]:
def minhash_signature(kgram_set, hash_funcs):
    signature = []
    
    for a, b in hash_funcs:
        min_val = float('inf')
        
        for g in kgram_set:
            x = hash_kgram(g)
            h = (a * x + b) % m
            
            if h < min_val:
                min_val = h
                
        signature.append(min_val)
        
    return signature

In [20]:
def signature_similarity(sig1, sig2):
    matches = sum(1 for i in range(len(sig1)) if sig1[i] == sig2[i])
    return matches / len(sig1)

In [21]:
exact_sim = jaccard(char3["D1"], char3["D2"])
print("Exact Jaccard Similarity (3-grams):", exact_sim)

Exact Jaccard Similarity (3-grams): 0.977979274611399


In [22]:
t_values = [20, 60, 150, 300, 600]

for t in t_values:
    hash_funcs = generate_hash_functions(t)
    
    sig1 = minhash_signature(char3["D1"], hash_funcs)
    sig2 = minhash_signature(char3["D2"], hash_funcs)
    
    approx_sim = signature_similarity(sig1, sig2)
    
    print(f"t = {t} → Approx Similarity: {approx_sim}")

t = 20 → Approx Similarity: 0.95
t = 60 → Approx Similarity: 0.9666666666666667
t = 150 → Approx Similarity: 0.96
t = 300 → Approx Similarity: 0.9766666666666667
t = 600 → Approx Similarity: 0.9833333333333333


## Question 3: Locality Sensitive Hashing
LSH banding technique is used to detect document pairs with similarity greater than a threshold.

In [23]:
t = 160
hash_funcs_160 = generate_hash_functions(t)

In [24]:
signatures = {}

for name in documents:
    signatures[name] = minhash_signature(char3[name], hash_funcs_160)

print("Signatures created for all documents.")

Signatures created for all documents.


In [26]:
def threshold_estimate(r, b):
    return (1/b)**(1/r)

pairs_rb = [(5,32), (8,20), (10,16), (16,10), (20,8), (32,5), (40,4)]

for r, b in pairs_rb:
    print(f"r={r}, b={b} → approx threshold ≈ {threshold_estimate(r,b)}")

r=5, b=32 → approx threshold ≈ 0.5
r=8, b=20 → approx threshold ≈ 0.6876560219336321
r=10, b=16 → approx threshold ≈ 0.757858283255199
r=16, b=10 → approx threshold ≈ 0.8659643233600653
r=20, b=8 → approx threshold ≈ 0.9012504626108302
r=32, b=5 → approx threshold ≈ 0.9509489152433014
r=40, b=4 → approx threshold ≈ 0.9659363289248456


In [27]:
r = 8
b = 20

def lsh_candidate_pairs(signatures, r, b):
    buckets = [{} for _ in range(b)]
    
    for doc_name, sig in signatures.items():
        for band in range(b):
            start = band * r
            end = start + r
            band_slice = tuple(sig[start:end])
            
            if band_slice not in buckets[band]:
                buckets[band][band_slice] = []
                
            buckets[band][band_slice].append(doc_name)
    
    candidate_pairs = set()
    
    for band in buckets:
        for docs in band.values():
            if len(docs) > 1:
                for pair in itertools.combinations(docs, 2):
                    candidate_pairs.add(tuple(sorted(pair)))
    
    return candidate_pairs

In [28]:
candidates = lsh_candidate_pairs(signatures, r, b)

print("Candidate pairs found by LSH:")
print(candidates)

Candidate pairs found by LSH:
{('D1', 'D2'), ('D3', 'D4')}


In [29]:
def lsh_probability(s, r, b):
    return 1 - (1 - s**r)**b

pairs = list(itertools.combinations(documents.keys(), 2))

print("LSH Probability for each pair:\n")

for d1, d2 in pairs:
    s = jaccard(char3[d1], char3[d2])
    prob = lsh_probability(s, r, b)
    print(f"{d1}-{d2}: similarity={s:.4f}, probability={prob:.6f}")

LSH Probability for each pair:

D1-D2: similarity=0.9780, probability=1.000000
D1-D3: similarity=0.5804, probability=0.228224
D1-D4: similarity=0.3051, probability=0.001500
D2-D3: similarity=0.5680, probability=0.195880
D2-D4: similarity=0.3059, probability=0.001532
D3-D4: similarity=0.3121, probability=0.001800


## Question 4: MinHash on MovieLens Dataset
The MovieLens 100k dataset is used to compute user similarity based on movies rated.

In [30]:
user_movies = {}

with open("ml-100k/u.data", "r") as f:
    for line in f:
        user_id, movie_id, rating, timestamp = line.strip().split("\t")
        
        user_id = int(user_id)
        movie_id = int(movie_id)
        
        if user_id not in user_movies:
            user_movies[user_id] = set()
        
        user_movies[user_id].add(movie_id)

print("Total users:", len(user_movies))

Total users: 943


In [31]:
import itertools
import time

start_time = time.time()

exact_pairs = set()

user_ids = list(user_movies.keys())

for u1, u2 in itertools.combinations(user_ids, 2):
    set1 = user_movies[u1]
    set2 = user_movies[u2]
    
    sim = len(set1 & set2) / len(set1 | set2)
    
    if sim >= 0.5:
        exact_pairs.add((u1, u2))

end_time = time.time()

print("Number of user pairs with similarity >= 0.5:", len(exact_pairs))
print("Time taken (seconds):", end_time - start_time)

Number of user pairs with similarity >= 0.5: 10
Time taken (seconds): 2.146466016769409


In [32]:
all_movies = set()

for movies in user_movies.values():
    all_movies.update(movies)

print("Total unique movies:", len(all_movies))

Total unique movies: 1682


In [33]:
def generate_user_minhash(user_movies, t, max_movie_id):
    m = 20011  # large prime
    
    # Generate hash functions
    hash_funcs = []
    for _ in range(t):
        a = random.randint(1, m-1)
        b = random.randint(0, m-1)
        hash_funcs.append((a, b))
    
    signatures = {}
    
    for user, movies in user_movies.items():
        sig = []
        
        for a, b in hash_funcs:
            min_hash = float('inf')
            
            for movie in movies:
                h = (a * movie + b) % m
                if h < min_hash:
                    min_hash = h
            
            sig.append(min_hash)
        
        signatures[user] = sig
    
    return signatures

In [34]:
def signature_similarity(sig1, sig2):
    matches = sum(1 for i in range(len(sig1)) if sig1[i] == sig2[i])
    return matches / len(sig1)

In [35]:
def evaluate_minhash(user_movies, exact_pairs, t):
    signatures = generate_user_minhash(user_movies, t, 1682)
    
    estimated_pairs = set()
    
    user_ids = list(user_movies.keys())
    
    for u1, u2 in itertools.combinations(user_ids, 2):
        sim = signature_similarity(signatures[u1], signatures[u2])
        
        if sim >= 0.5:
            estimated_pairs.add((u1, u2))
    
    false_positives = len(estimated_pairs - exact_pairs)
    false_negatives = len(exact_pairs - estimated_pairs)
    
    return false_positives, false_negatives

In [36]:
fp, fn = evaluate_minhash(user_movies, exact_pairs, 50)

print("For t=50:")
print("False Positives:", fp)
print("False Negatives:", fn)

For t=50:
False Positives: 159
False Negatives: 1


In [37]:
def average_runs(user_movies, exact_pairs, t, runs=5):
    total_fp = 0
    total_fn = 0
    
    for i in range(runs):
        print(f"Run {i+1} for t={t}")
        fp, fn = evaluate_minhash(user_movies, exact_pairs, t)
        
        total_fp += fp
        total_fn += fn
    
    avg_fp = total_fp / runs
    avg_fn = total_fn / runs
    
    return avg_fp, avg_fn

In [38]:
avg_fp_50, avg_fn_50 = average_runs(user_movies, exact_pairs, 50)

print("\nAverage Results for t=50:")
print("Average False Positives:", avg_fp_50)
print("Average False Negatives:", avg_fn_50)

Run 1 for t=50
Run 2 for t=50
Run 3 for t=50
Run 4 for t=50
Run 5 for t=50

Average Results for t=50:
Average False Positives: 156.4
Average False Negatives: 2.4


In [39]:
avg_fp_100, avg_fn_100 = average_runs(user_movies, exact_pairs, 100)

print("\nAverage Results for t=100:")
print("Average False Positives:", avg_fp_100)
print("Average False Negatives:", avg_fn_100)

Run 1 for t=100
Run 2 for t=100
Run 3 for t=100
Run 4 for t=100
Run 5 for t=100

Average Results for t=100:
Average False Positives: 28.8
Average False Negatives: 2.6


In [40]:
avg_fp_200, avg_fn_200 = average_runs(user_movies, exact_pairs, 200)

print("\nAverage Results for t=200:")
print("Average False Positives:", avg_fp_200)
print("Average False Negatives:", avg_fn_200)

Run 1 for t=200
Run 2 for t=200
Run 3 for t=200
Run 4 for t=200
Run 5 for t=200

Average Results for t=200:
Average False Positives: 11.0
Average False Negatives: 1.4


## Question 5: LSH on MovieLens Dataset
LSH is applied on user MinHash signatures to efficiently find similar users.

In [41]:
def compute_exact_pairs(user_movies, threshold):
    pairs = set()
    user_ids = list(user_movies.keys())
    
    for u1, u2 in itertools.combinations(user_ids, 2):
        set1 = user_movies[u1]
        set2 = user_movies[u2]
        
        sim = len(set1 & set2) / len(set1 | set2)
        
        if sim >= threshold:
            pairs.add((u1, u2))
    
    return pairs

exact_pairs_06 = compute_exact_pairs(user_movies, 0.6)
exact_pairs_08 = compute_exact_pairs(user_movies, 0.8)

print("Pairs >= 0.6:", len(exact_pairs_06))
print("Pairs >= 0.8:", len(exact_pairs_08))

Pairs >= 0.6: 3
Pairs >= 0.8: 1


In [42]:
def lsh_candidates_user(signatures, r, b):
    buckets = [{} for _ in range(b)]
    
    for user, sig in signatures.items():
        for band in range(b):
            start = band * r
            end = start + r
            band_slice = tuple(sig[start:end])
            
            if band_slice not in buckets[band]:
                buckets[band][band_slice] = []
                
            buckets[band][band_slice].append(user)
    
    candidate_pairs = set()
    
    for band in buckets:
        for users in band.values():
            if len(users) > 1:
                for pair in itertools.combinations(users, 2):
                    candidate_pairs.add(tuple(sorted(pair)))
    
    return candidate_pairs

In [43]:
def evaluate_lsh(user_movies, exact_pairs, t, r, b):
    signatures = generate_user_minhash(user_movies, t, 1682)
    
    candidates = lsh_candidates_user(signatures, r, b)
    
    false_positives = len(candidates - exact_pairs)
    false_negatives = len(exact_pairs - candidates)
    
    return false_positives, false_negatives

In [44]:
def average_lsh_runs(user_movies, exact_pairs, t, r, b, runs=5):
    total_fp = 0
    total_fn = 0
    
    for i in range(runs):
        print(f"Run {i+1} (t={t}, r={r}, b={b})")
        fp, fn = evaluate_lsh(user_movies, exact_pairs, t, r, b)
        
        total_fp += fp
        total_fn += fn
    
    return total_fp / runs, total_fn / runs

In [45]:
avg_fp_50_lsh_06, avg_fn_50_lsh_06 = average_lsh_runs(
    user_movies, exact_pairs_06, 50, 5, 10
)

print("\nLSH Results (t=50, r=5, b=10, threshold=0.6)")
print("Average False Positives:", avg_fp_50_lsh_06)
print("Average False Negatives:", avg_fn_50_lsh_06)

Run 1 (t=50, r=5, b=10)
Run 2 (t=50, r=5, b=10)
Run 3 (t=50, r=5, b=10)
Run 4 (t=50, r=5, b=10)
Run 5 (t=50, r=5, b=10)

LSH Results (t=50, r=5, b=10, threshold=0.6)
Average False Positives: 667.0
Average False Negatives: 0.8


In [46]:
avg_fp_100_lsh_06, avg_fn_100_lsh_06 = average_lsh_runs(
    user_movies, exact_pairs_06, 100, 5, 20
)

print("\nLSH Results (t=100, r=5, b=20, threshold=0.6)")
print("Average False Positives:", avg_fp_100_lsh_06)
print("Average False Negatives:", avg_fn_100_lsh_06)

Run 1 (t=100, r=5, b=20)
Run 2 (t=100, r=5, b=20)
Run 3 (t=100, r=5, b=20)
Run 4 (t=100, r=5, b=20)
Run 5 (t=100, r=5, b=20)

LSH Results (t=100, r=5, b=20, threshold=0.6)
Average False Positives: 1439.2
Average False Negatives: 0.2


In [47]:
avg_fp_200_lsh_06_r5, avg_fn_200_lsh_06_r5 = average_lsh_runs(
    user_movies, exact_pairs_06, 200, 5, 40
)

print("\nLSH Results (t=200, r=5, b=40, threshold=0.6)")
print("Average False Positives:", avg_fp_200_lsh_06_r5)
print("Average False Negatives:", avg_fn_200_lsh_06_r5)

Run 1 (t=200, r=5, b=40)
Run 2 (t=200, r=5, b=40)
Run 3 (t=200, r=5, b=40)
Run 4 (t=200, r=5, b=40)
Run 5 (t=200, r=5, b=40)

LSH Results (t=200, r=5, b=40, threshold=0.6)
Average False Positives: 2189.2
Average False Negatives: 0.0


In [48]:
avg_fp_200_lsh_06_r10, avg_fn_200_lsh_06_r10 = average_lsh_runs(
    user_movies, exact_pairs_06, 200, 10, 20
)

print("\nLSH Results (t=200, r=10, b=20, threshold=0.6)")
print("Average False Positives:", avg_fp_200_lsh_06_r10)
print("Average False Negatives:", avg_fn_200_lsh_06_r10)

Run 1 (t=200, r=10, b=20)
Run 2 (t=200, r=10, b=20)
Run 3 (t=200, r=10, b=20)
Run 4 (t=200, r=10, b=20)
Run 5 (t=200, r=10, b=20)

LSH Results (t=200, r=10, b=20, threshold=0.6)
Average False Positives: 4.4
Average False Negatives: 1.0


In [49]:
avg_fp_50_lsh_08, avg_fn_50_lsh_08 = average_lsh_runs(
    user_movies, exact_pairs_08, 50, 5, 10
)

print("\nLSH Results (t=50, r=5, b=10, threshold=0.8)")
print("Average False Positives:", avg_fp_50_lsh_08)
print("Average False Negatives:", avg_fn_50_lsh_08)

Run 1 (t=50, r=5, b=10)
Run 2 (t=50, r=5, b=10)
Run 3 (t=50, r=5, b=10)
Run 4 (t=50, r=5, b=10)
Run 5 (t=50, r=5, b=10)

LSH Results (t=50, r=5, b=10, threshold=0.8)
Average False Positives: 553.0
Average False Negatives: 0.0


In [50]:
avg_fp_100_lsh_08, avg_fn_100_lsh_08 = average_lsh_runs(
    user_movies, exact_pairs_08, 100, 5, 20
)

print("\nLSH Results (t=100, r=5, b=20, threshold=0.8)")
print("Average False Positives:", avg_fp_100_lsh_08)
print("Average False Negatives:", avg_fn_100_lsh_08)

Run 1 (t=100, r=5, b=20)
Run 2 (t=100, r=5, b=20)
Run 3 (t=100, r=5, b=20)
Run 4 (t=100, r=5, b=20)
Run 5 (t=100, r=5, b=20)

LSH Results (t=100, r=5, b=20, threshold=0.8)
Average False Positives: 1214.8
Average False Negatives: 0.0


In [51]:
avg_fp_200_lsh_08_r5, avg_fn_200_lsh_08_r5 = average_lsh_runs(
    user_movies, exact_pairs_08, 200, 5, 40
)

print("\nLSH Results (t=200, r=5, b=40, threshold=0.8)")
print("Average False Positives:", avg_fp_200_lsh_08_r5)
print("Average False Negatives:", avg_fn_200_lsh_08_r5)

Run 1 (t=200, r=5, b=40)
Run 2 (t=200, r=5, b=40)
Run 3 (t=200, r=5, b=40)
Run 4 (t=200, r=5, b=40)
Run 5 (t=200, r=5, b=40)

LSH Results (t=200, r=5, b=40, threshold=0.8)
Average False Positives: 2429.6
Average False Negatives: 0.0


In [52]:
avg_fp_200_lsh_08_r10, avg_fn_200_lsh_08_r10 = average_lsh_runs(
    user_movies, exact_pairs_08, 200, 10, 20
)

print("\nLSH Results (t=200, r=10, b=20, threshold=0.8)")
print("Average False Positives:", avg_fp_200_lsh_08_r10)
print("Average False Negatives:", avg_fn_200_lsh_08_r10)

Run 1 (t=200, r=10, b=20)
Run 2 (t=200, r=10, b=20)
Run 3 (t=200, r=10, b=20)
Run 4 (t=200, r=10, b=20)
Run 5 (t=200, r=10, b=20)

LSH Results (t=200, r=10, b=20, threshold=0.8)
Average False Positives: 3.0
Average False Negatives: 0.0
